# Public PancDB scRNA-seq — QC, Doublet Removal, and Mesenchymal Sub-clustering
## ND and T2D Adult Donors — Full Preprocessing Pipeline

**Input:** `PANCDB_public_NDT2D_Adult.h5ad` → `adata_2`
(Public HPAP/PancDB pancreatic scRNA-seq, adult donors,
pre-labeled broad cell types, ND and T2D conditions)

**Outputs:**
- `pancdb_nd_t2d_cleaned.h5ad` → `pancdb_adata_clean093109`
  (QC-filtered, doublet-removed, donor-cleaned full object)
- `pancdbmesendo_nd_t2d_cleaned.h5ad` → `mesenchymal_endopancdb`
  (Mesenchymal + endothelial cells only)
- `pancdbmes_subclusteredfiltered_nd_t2d_cleaned.h5ad` → `mesenchymalf`
  (Final mesenchymal object with MSL1/MSL2 labels)
- Per-cluster DEG tables: MSL1 and MSL2 T2D vs Control

---

### Pipeline Overview

**PART 1 — QC Metric Computation**

1. Load full public HPAP/PancDB adult ND+T2D object
2. Compute per-cell QC metrics manually (sparse-safe):
   - `total_counts` — total UMI counts
   - `n_genes` — genes detected per cell
   - `pct_mt` — percent mitochondrial reads (MT- prefix)
3. Compute composite QC score:
   - z-score(total_counts) + z-score(n_genes) − z-score(pct_mt)
   - Safe z-score function handles zero-variance edge cases
4. QC visualization per donor:
   - Per-donor histogram panels (total_counts, n_genes, pct_mt)
   - Overlaid KDE density curves per donor (colored by donor)
   - Per-donor QC score KDE overlay

**PART 2 — Donor Filtering and Adult Subset**

5. Filter to adult donors only (age ≥ 18)
6. Filter to Control and T2D disease states only
7. Per-donor total_counts histograms inspected individually
   for outlier donors (HPAP109, HPAP093, HPAP067, HPAP027)
8. Outlier donors removed iteratively based on QC distributions:
   - HPAP093 removed (low quality)
   - HPAP109 removed (outlier distribution)
   - HPAP067 removed (outlier distribution)
   - HPAP027 removed (outlier distribution)
9. Cells per sample barplot generated before and after filtering

**PART 3 — Doublet Detection**

10. Normalize + log1p transform on filtered object
11. Run doubletdetection.BoostClassifier:
    - n_iters=10, leiden clustering, standard_scaling=True
    - pseudocount=0.1, p_thresh=1e-16, voter_thresh=0.5
12. Doublet score distribution plotted (histogram with stats)
13. Doublets removed → `pancdb_adata_clean`
14. Cells per sample re-checked after doublet removal

**PART 4 — Concord Integration and UMAP**

15. Select top 2000 variable features (Seurat v3 flavor)
16. Concord batch integration (domain_key = donor_id)
17. UMAP computed from Concord latent space (n_neighbors=15,
    min_dist=0.1, euclidean metric)
18. Remove "unknown" cell type labels
19. UMAP visualized: donor_id, cell_type, disease_state, age
20. Final cleaned object saved as pancdb_nd_t2d_cleaned.h5ad

**PART 5 — Mesenchymal/Endothelial Subset**

21. Subset mesenchymal + endothelial cells → `mesenchymal_endopancdb`
22. DEG: T2D vs Control separately for:
    - Endothelial cells (Wilcoxon, use_raw=False)
    - Mesenchymal cells (Wilcoxon, use_raw=False)
23. Results exported as CSV per cell type
24. Marker gene UMAPs for mesenchymal identity confirmation:
    S100A6, LUM, COL6A3, DCN, RGS5, ADIRF, C11orf96,
    ADAMTS4, TFPI, FABP4, COL1A2

**PART 6 — Mesenchymal Sub-clustering → MSL1/MSL2**

25. Load mesenchymal-only object → `mesenchymalf`
26. Leiden clustering (res=0.1) on existing neighbor graph
27. DEG per Leiden cluster to assess biological identity
28. Cluster 6 removed (low quality / ambiguous)
29. DEG re-run after cluster 6 removal
30. Clusters merged into two final states:
    - Leiden 0,2,3,4,5 → MSL1 (fibroblast-like)
    - Leiden 1 → MSL2 (contractile/pericyte-like)
31. DEG: T2D vs Control per state (MSL1 and MSL2 separately)
    Wilcoxon, use_raw=False, contamination genes removed
32. Final object saved as:
    pancdbmes_subclusteredfiltered_nd_t2d_cleaned.h5ad

---

### Mesenchymal Cluster → Final Label Mapping

| Leiden Clusters | Final Label | Biological Identity |
|---|---|---|
| 0, 2, 3, 4, 5 | MSL1 | Fibroblast-like (LUM, DCN, COL1A2 high) |
| 1 | MSL2 | Contractile/pericyte-like (RGS5, ADIRF high) |
| 6 | Removed | Low quality / ambiguous |

---

### Donors Removed During QC

| Donor | Reason |
|---|---|
| HPAP093 | Outlier QC distribution |
| HPAP109 | Outlier count distribution |
| HPAP067 | Outlier count distribution |
| HPAP027 | Outlier count distribution |

---

### QC Metrics Computed

| Metric | Description |
|---|---|
| `total_counts` | Total UMI counts per cell |
| `n_genes` | Genes detected per cell |
| `pct_mt` | % mitochondrial counts |
| `QC_score` | z(counts) + z(genes) − z(pct_mt) |
| `doublet` | Binary doublet call (BoostClassifier) |
| `doublet_score` | Continuous doublet probability |

---

### DEG Comparisons Run

| Comparison | Cell Type | Reference | Method |
|---|---|---|---|
| T2D vs Control | Endothelial cells | Control | Wilcoxon |
| T2D vs Control | Mesenchymal cells | Control | Wilcoxon |
| T2D vs Control | MSL1 | Control | Wilcoxon |
| T2D vs Control | MSL2 | Control | Wilcoxon |

Contamination genes removed before export:
PRSS1, PRSS2, CPA1, CLPS, CTRB1, CTRB2, CELA3A, REG1A, PLA2G1B

---

### Key Marker Genes Used for MSL Annotation

| Marker | Enriched In |
|---|---|
| LUM, DCN, COL1A2, COL6A3 | MSL1 (fibroblast-like) |
| RGS5, ADIRF, C11orf96 | MSL2 (contractile/pericyte-like) |
| S100A6, FABP4 | Adipocyte/stress markers (contamination check) |
| ADAMTS4, TFPI | Remodeling markers |

---

### Key Output Columns

| Column | Object | Description |
|---|---|---|
| `mesenchymal_leiden` | mesenchymalf | Raw Leiden clusters (res=0.1) |
| `mesenchymal_leiden` | mesenchymalf | Final MSL1/MSL2 labels after merging |
| `doublet` | pancdb_adata_clean | Binary doublet call |
| `doublet_score` | pancdb_adata_clean | Continuous doublet score |
| `disease_state` | all objects | Control / T2D |
| `donor_id` | all objects | HPAP donor ID |

In [ ]:
import concord as ccd
import scanpy as sc
import anndata as ad
import torch
import os
import pandas as pd 
import numpy as np
import seaborn as sns

In [ ]:
adata_2 = sc.read_h5ad('/Volumes/KM_2TB_SSD/Public_human_RNAdasets/PANCDB_public_NDT2D_Adult.h5ad')

In [ ]:
adata_2

In [ ]:
# =========================================================
# COMPUTE QC METRICS
# =========================================================

import scanpy as sc
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# total counts
# ---------------------------------------------------------

adata_2.obs["total_counts"] = np.array(
    adata_2.X.sum(axis=1)
).flatten()

# ---------------------------------------------------------
# number of genes detected
# ---------------------------------------------------------

adata_2.obs["n_genes"] = np.array(
    (adata_2.X > 0).sum(axis=1)
).flatten()

# ---------------------------------------------------------
# mitochondrial genes
# ---------------------------------------------------------

adata_2.var["mt"] = (
    adata_2.var_names
    .str.upper()
    .str.startswith("MT-")
)

mt_counts = np.array(
    adata_2[:, adata_2.var["mt"]].X.sum(axis=1)
).flatten()

adata_2.obs["pct_mt"] = (
    mt_counts
    /
    adata_2.obs["total_counts"]
) * 100

# =========================================================
# CHECK
# =========================================================

print(
    adata_2.obs[
        [
            "total_counts",
            "n_genes",
            "pct_mt"
        ]
    ].head()
)

In [ ]:
# =========================================================
# DONOR QC HISTOGRAMS
# =========================================================

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================================================
# SETTINGS
# =========================================================

DONOR_COL = "donor_id"

qc_metrics = [
    "total_counts",
    "n_genes",
    "pct_mt"
]

# =========================================================
# STYLE
# =========================================================

mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================================================
# PLOTS
# =========================================================

for metric in qc_metrics:

    print(f"\nPlotting: {metric}")

    donors = sorted(
        adata_2.obs[DONOR_COL].unique()
    )

    n = len(donors)

    fig, axes = plt.subplots(
        nrows=int(np.ceil(n / 3)),
        ncols=3,
        figsize=(7, 2.2 * int(np.ceil(n / 3)))
    )

    axes = np.array(axes).flatten()

    for i, donor in enumerate(donors):

        ax = axes[i]

        vals = adata_2.obs.loc[
            adata_2.obs[DONOR_COL] == donor,
            metric
        ]

        ax.hist(
            vals,
            bins=50
        )

        ax.set_title(
            donor,
            fontsize=7
        )

        ax.set_xlabel(metric)
        ax.set_ylabel("Cells")

    # remove empty panels
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()

    plt.savefig(
        f"/Users/kmeneses/Desktop/{metric}_donor_histograms.pdf",
        bbox_inches="tight"
    )

    plt.savefig(
        f"/Users/kmeneses/Desktop/{metric}_donor_histograms.svg",
        bbox_inches="tight"
    )

    plt.show()

print("✔ QC histograms complete")

In [ ]:
# =========================================================
# OVERLAID DONOR QC HISTOGRAMS
#
# each donor =
# one colored density line
#
# plots:
# - total_counts
# - n_genes
# - pct_mt
# =========================================================

import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import numpy as np

# =========================================================
# SETTINGS
# =========================================================

DONOR_COL = "donor_id"

qc_metrics = [
    "total_counts",
    "n_genes",
    "pct_mt"
]

OUT = "/Users/kmeneses/Desktop"

# =========================================================
# STYLE
# =========================================================

mpl.rcParams.update({

    "font.family": "Arial",
    "font.size": 8,
    "axes.linewidth": 1,

    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================================================
# COLORS
# =========================================================

donors = sorted(
    adata_2.obs[DONOR_COL].unique()
)

palette = sns.color_palette(
    "tab20",
    n_colors=len(donors)
)

donor_colors = dict(
    zip(donors, palette)
)

# =========================================================
# PLOTS
# =========================================================

for metric in qc_metrics:

    print(f"\nPlotting: {metric}")

    fig, ax = plt.subplots(
        figsize=(4,3)
    )

    # -----------------------------------------------------
    # plot each donor
    # -----------------------------------------------------

    for donor in donors:

        vals = adata_2.obs.loc[
            adata_2.obs[DONOR_COL] == donor,
            metric
        ]

        sns.kdeplot(
            vals,
            ax=ax,
            linewidth=1.5,
            label=donor,
            color=donor_colors[donor]
        )

    # -----------------------------------------------------
    # labels
    # -----------------------------------------------------

    ax.set_title(
        f"{metric} distributions",
        fontsize=9,
        fontweight="bold"
    )

    ax.set_xlabel(metric)
    ax.set_ylabel("Density")

    # -----------------------------------------------------
    # legend
    # -----------------------------------------------------

    ax.legend(
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        frameon=False,
        fontsize=6
    )

    sns.despine()

    plt.tight_layout()

    # -----------------------------------------------------
    # SAVE
    # -----------------------------------------------------

    plt.savefig(
        f"{OUT}/{metric}_overlap_density.pdf",
        bbox_inches="tight"
    )

    plt.savefig(
        f"{OUT}/{metric}_overlap_density.svg",
        bbox_inches="tight"
    )

    plt.show()

print("\n✔ overlap QC plots complete")

In [ ]:
# =========================================================
# COMPUTE QC METRICS + QC SCORE
# =========================================================

import numpy as np
import pandas as pd
from scipy.stats import zscore
from scipy import sparse

# =========================================================
# TOTAL COUNTS
# =========================================================

if sparse.issparse(adata_2.X):

    adata_2.obs["total_counts"] = np.array(
        adata_2.X.sum(axis=1)
    ).flatten()

else:

    adata_2.obs["total_counts"] = (
        adata_2.X.sum(axis=1)
    )

# =========================================================
# N GENES
# =========================================================

if sparse.issparse(adata_2.X):

    adata_2.obs["n_genes"] = np.array(
        (adata_2.X > 0).sum(axis=1)
    ).flatten()

else:

    adata_2.obs["n_genes"] = (
        (adata_2.X > 0).sum(axis=1)
    )

# =========================================================
# MT GENES
# =========================================================

adata_2.var["mt"] = (
    adata_2.var_names
    .str.upper()
    .str.startswith("MT-")
)

if adata_2.var["mt"].sum() > 0:

    mt_counts = np.array(
        adata_2[
            :,
            adata_2.var["mt"]
        ].X.sum(axis=1)
    ).flatten()

    adata_2.obs["pct_mt"] = (
        mt_counts
        /
        adata_2.obs["total_counts"]
    ) * 100

else:

    adata_2.obs["pct_mt"] = 0

# =========================================================
# QC SCORE
# =========================================================

adata_2.obs["QC_score"] = (

    zscore(
        adata_2.obs["total_counts"]
    )

    +

    zscore(
        adata_2.obs["n_genes"]
    )

    -

    zscore(
        adata_2.obs["pct_mt"]
    )

)

# =========================================================
# CHECK
# =========================================================

print(
    adata_2.obs[
        [
            "total_counts",
            "n_genes",
            "pct_mt",
            "QC_score"
        ]
    ].head()
)

print("\nQC score summary:")
print(
    adata_2.obs["QC_score"].describe()
)

In [ ]:
from scipy.stats import zscore
import numpy as np

# =========================================================
# SAFE ZSCORE FUNCTION
# =========================================================

def safe_zscore(x):

    if np.std(x) == 0:

        return np.zeros(len(x))

    return zscore(x)

# =========================================================
# QC SCORE
# =========================================================

adata_2.obs["QC_score"] = (

    safe_zscore(
        adata_2.obs["total_counts"]
    )

    +

    safe_zscore(
        adata_2.obs["n_genes"]
    )

    -

    safe_zscore(
        adata_2.obs["pct_mt"]
    )

)

# =========================================================
# CHECK
# =========================================================

print(
    adata_2.obs["QC_score"].describe()
)

In [ ]:
# =========================================================
# QC DISTRIBUTION PER DONOR
# OVERLAID KDE CURVES
# =========================================================

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

# =========================================================
# SETTINGS
# =========================================================

DONOR_COL = "donor_id"

metric = "QC_score"   # <- change if needed

OUT = "/Users/kmeneses/Desktop"

# =========================================================
# STYLE
# =========================================================

mpl.rcParams.update({

    "font.family": "Arial",
    "font.size": 8,

    "axes.linewidth": 1,

    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================================================
# DONORS
# =========================================================

donors = sorted(
    adata_2.obs[DONOR_COL].unique()
)

# =========================================================
# COLORS
# =========================================================

palette = sns.color_palette(
    "tab20",
    n_colors=len(donors)
)

# =========================================================
# PLOT
# =========================================================

fig, ax = plt.subplots(
    figsize=(6,4)
)

for donor, color in zip(donors, palette):

    vals = adata_2.obs.loc[
        adata_2.obs[DONOR_COL] == donor,
        metric
    ]

    sns.kdeplot(
        vals,
        ax=ax,
        linewidth=1.5,
        alpha=0.7,
        color=color,
        label=donor
    )

# =========================================================
# LABELS
# =========================================================

ax.set_title(
    "QC distribution per donor",
    fontsize=10,
    fontweight="normal"
)

ax.set_xlabel("QC score")
ax.set_ylabel("Density")

# =========================================================
# LEGEND
# =========================================================

ax.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
    fontsize=7
)

sns.despine()

plt.tight_layout()

# =========================================================
# SAVE
# =========================================================

plt.savefig(
    f"{OUT}/QC_distribution_per_donor.pdf",
    bbox_inches="tight"
)

plt.savefig(
    f"{OUT}/QC_distribution_per_donor.svg",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

print("✔ done")

In [ ]:
color_by = ['disease_state', 'age'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    adata, basis='X_umap', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=10, legend_loc='upper right')

In [ ]:
import pandas as pd

# convert age column to numeric
adata.obs["age"] = pd.to_numeric(
    adata.obs["age"],
    errors="coerce"
)

# keep only samples age 18+
pancdb_adata_both = adata[
    adata.obs["age"] >= 18
].copy()

In [ ]:
cell_type = ['T2D', 'Control']
pancdb_adata_both = pancdb_adata_both[pancdb_adata_both.obs["disease_state"].isin(cell_type)
].copy()

In [ ]:
color_by = ['disease_state', 'age'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    pancdb_adata_both, basis='X_umap', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=10, legend_loc='upper right')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import scanpy as sc

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "font.weight": "normal", 
    "axes.titleweight": "normal",   # 🔥 no bold
    "axes.labelweight": "normal", 
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})
plt.rcParams["figure.figsize"] = (2, 2)
# =========================
# PLOT (let scanpy build grid)
# =========================
fig = sc.pl.embedding(
    adata,
    basis="Concord_UMAP",
    color=['donor_id', 'cell_type'],
    ncols=1,
    size=1,
    wspace=0.1,                 # 🔥 smaller = lighter file
    frameon=True,
    legend_loc='right',
    show=False,
    return_fig=True           # 🔥 IMPORTANT
)

# =========================
# 🔥 RASTERIZE POINTS (KEY FIX)
# =========================
for ax in fig.axes:
    for coll in ax.collections:
        coll.set_rasterized(True)

# =========================
# CLEAN AXES + BORDER
# =========================
for ax in fig.axes:
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.title.set_fontweight("normal")
    ax.xaxis.label.set_fontweight("normal")
    ax.yaxis.label.set_fontweight("normal")
    
    for spine in ax.spines.values():
        spine.set_linewidth(1)

# =========================
# EXPORT
# =========================
out_base = "/Users/kmeneses/Desktop/Figure4_plots/SI/pandc_ht2dadault_UMAP_db"

fig.savefig(f"{out_base}.pdf", dpi=600, bbox_inches="tight")  # 🔥 BEST for Illustrator
fig.savefig(f"{out_base}.svg", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import scanpy as sc

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "font.weight": "normal", 
    "axes.titleweight": "normal",   # 🔥 no bold
    "axes.labelweight": "normal", 
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})
plt.rcParams["figure.figsize"] = (2, 2)
# =========================
# PLOT (let scanpy build grid)
# =========================
fig = sc.pl.embedding(
    adata,
    basis="Concord_UMAP",
    color=['disease_state', 'age'],
    ncols=1,
    size=1,
    wspace=0.1,                 # 🔥 smaller = lighter file
    frameon=True,
    legend_loc='right',
    show=False,
    return_fig=True           # 🔥 IMPORTANT
)

# =========================
# 🔥 RASTERIZE POINTS (KEY FIX)
# =========================
for ax in fig.axes:
    for coll in ax.collections:
        coll.set_rasterized(True)

# =========================
# CLEAN AXES + BORDER
# =========================
for ax in fig.axes:
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.title.set_fontweight("normal")
    ax.xaxis.label.set_fontweight("normal")
    ax.yaxis.label.set_fontweight("normal")
    
    for spine in ax.spines.values():
        spine.set_linewidth(1)

# =========================
# EXPORT
# =========================
out_base = "/Users/kmeneses/Desktop/Figure4_plots/SI/pandc_ht2dadault_UMAP_dbdsage"

fig.savefig(f"{out_base}.pdf", dpi=600, bbox_inches="tight")  # 🔥 BEST for Illustrator
fig.savefig(f"{out_base}.svg", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
sc.pp.calculate_qc_metrics(
    pancdb_adata_both,
    inplace=True
)

In [ ]:
sc.pl.violin(
    pancdb_adata_both,
    ["n_genes_by_counts"],
    groupby="donor_id",
    jitter=0.1,
    rotation=90,
)

In [ ]:
sc.pl.violin(
    pancdb_adata_both,
    ["total_counts"],
    groupby="donor_id",
    jitter=0.1,
    rotation=90,
)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# =====================================================
# SETTINGS
# =====================================================
DONOR = "HPAP109"   # change donor

QC_COL = "total_counts"

# =====================================================
# SUBSET DONOR
# =====================================================
adata_d = pancdb_adata_both[
    pancdb_adata_both.obs["donor_id"] == DONOR
].copy()

print(adata_d.shape)

# =====================================================
# STYLE
# =====================================================
mpl.rcParams.update({
    "font.family": "Arial",
    "svg.fonttype": "none",
    "pdf.fonttype": 42
})

# =====================================================
# PLOT DISTRIBUTION
# =====================================================
fig, ax = plt.subplots(figsize=(3,2))

sns.histplot(
    adata_d.obs[QC_COL],
    bins=50,
    kde=True,
    ax=ax
)

ax.set_xlabel(QC_COL)
ax.set_ylabel("Cells")
ax.set_title(DONOR)

sns.despine()

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# =====================================================
# SETTINGS
# =====================================================
DONOR = "HPAP109"   # change donor
QC_COL = "total_counts"

# =====================================================
# SUBSET DONOR
# =====================================================
adata_d = adata_2[
    adata_2.obs["donor_id"] == DONOR
].copy()

print(adata_d.shape)

# =====================================================
# STYLE
# =====================================================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.linewidth": 1,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =====================================================
# SMALL HISTOGRAM
# =====================================================
fig, ax = plt.subplots(figsize=(2.2,1.8))

sns.histplot(
    adata_d.obs[QC_COL],
    bins=50,
    kde=True,
    ax=ax,
    linewidth=0
)

ax.set_xlabel(QC_COL)
ax.set_ylabel("Cells")
ax.set_title(DONOR, fontsize=7)

sns.despine()

plt.tight_layout()

# =====================================================
# SAVE HIGH-RES
# =====================================================
fig.savefig(
    f"/Users/kmeneses/Desktop/{DONOR}_{QC_COL}_histogramT2DHPAP109.svg",
    format="svg",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    f"/Users/kmeneses/Desktop/{DONOR}_{QC_COL}_histogram.pdf",
    format="pdf",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# =====================================================
# SETTINGS
# =====================================================
DONOR = "HPAP093"   # change donor

QC_COL = "total_counts"

# =====================================================
# SUBSET DONOR
# =====================================================
adata_d = pancdb_adata_both[
    pancdb_adata_both.obs["donor_id"] == DONOR
].copy()

print(adata_d.shape)

# =====================================================
# STYLE
# =====================================================
mpl.rcParams.update({
    "font.family": "Arial",
    "svg.fonttype": "none",
    "pdf.fonttype": 42
})

# =====================================================
# PLOT DISTRIBUTION
# =====================================================
fig, ax = plt.subplots(figsize=(3,2))

sns.histplot(
    adata_d.obs[QC_COL],
    bins=50,
    kde=True,
    ax=ax
)

ax.set_xlabel(QC_COL)
ax.set_ylabel("Cells")
ax.set_title(DONOR)

sns.despine()

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# =====================================================
# SETTINGS
# =====================================================
DONOR = "HPAP051"   # change donor

QC_COL = "total_counts"

# =====================================================
# SUBSET DONOR
# =====================================================
adata_d = pancdb_adata_both[
    pancdb_adata_both.obs["donor_id"] == DONOR
].copy()

print(adata_d.shape)

# =====================================================
# STYLE
# =====================================================
mpl.rcParams.update({
    "font.family": "Arial",
    "svg.fonttype": "none",
    "pdf.fonttype": 42
})

# =====================================================
# PLOT DISTRIBUTION
# =====================================================
fig, ax = plt.subplots(figsize=(3,2))

sns.histplot(
    adata_d.obs[QC_COL],
    bins=50,
    kde=True,
    ax=ax
)

ax.set_xlabel(QC_COL)
ax.set_ylabel("Cells")
ax.set_title(DONOR)

sns.despine()

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# =====================================================
# SETTINGS
# =====================================================
DONOR = "HPAP067"   # change donor

QC_COL = "total_counts"

# =====================================================
# SUBSET DONOR
# =====================================================
adata_d = pancdb_adata_both[
    pancdb_adata_both.obs["donor_id"] == DONOR
].copy()

print(adata_d.shape)

# =====================================================
# STYLE
# =====================================================
mpl.rcParams.update({
    "font.family": "Arial",
    "svg.fonttype": "none",
    "pdf.fonttype": 42
})

# =====================================================
# PLOT DISTRIBUTION
# =====================================================
fig, ax = plt.subplots(figsize=(3,2))

sns.histplot(
    adata_d.obs[QC_COL],
    bins=50,
    kde=True,
    ax=ax
)

ax.set_xlabel(QC_COL)
ax.set_ylabel("Cells")
ax.set_title(DONOR)

sns.despine()

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# =====================================================
# SETTINGS
# =====================================================
DONOR = "HPAP070"   # change donor

QC_COL = "total_counts"

# =====================================================
# SUBSET DONOR
# =====================================================
adata_d = pancdb_adata_both[
    pancdb_adata_both.obs["donor_id"] == DONOR
].copy()

print(adata_d.shape)

# =====================================================
# STYLE
# =====================================================
mpl.rcParams.update({
    "font.family": "Arial",
    "svg.fonttype": "none",
    "pdf.fonttype": 42
})

# =====================================================
# PLOT DISTRIBUTION
# =====================================================
fig, ax = plt.subplots(figsize=(3,2))

sns.histplot(
    adata_d.obs[QC_COL],
    bins=50,
    kde=True,
    ax=ax
)

ax.set_xlabel(QC_COL)
ax.set_ylabel("Cells")
ax.set_title(DONOR)

sns.despine()

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

# =====================================================
# SETTINGS
# =====================================================
DONOR = "HPAP106"   # change donor

QC_COL = "total_counts"

# =====================================================
# SUBSET DONOR
# =====================================================
adata_d = pancdb_adata_both[
    pancdb_adata_both.obs["donor_id"] == DONOR
].copy()

print(adata_d.shape)

# =====================================================
# STYLE
# =====================================================
mpl.rcParams.update({
    "font.family": "Arial",
    "svg.fonttype": "none",
    "pdf.fonttype": 42
})

# =====================================================
# PLOT DISTRIBUTION
# =====================================================
fig, ax = plt.subplots(figsize=(3,2))

sns.histplot(
    adata_d.obs[QC_COL],
    bins=50,
    kde=True,
    ax=ax
)

ax.set_xlabel(QC_COL)
ax.set_ylabel("Cells")
ax.set_title(DONOR)

sns.despine()

plt.tight_layout()
plt.show()

In [ ]:
import scanpy as sc

# 1) Preserve raw counts if not already stored
pancdb_adata_both.raw = pancdb_adata_both.copy()   # keeps raw counts in .raw.X
# 2) Normalize total counts per cell
sc.pp.normalize_total(pancdb_adata_both, target_sum=1e4)

# 3) Log1p transform
sc.pp.log1p(pancdb_adata_both)

# 4) Store the log1p matrix in a layer
pancdb_adata_both.layers["log1p"] = pancdb_adata_both.X.copy()

In [ ]:
# Define clusters to remove
clusters_to_remove = ["HPAP093"]  # Example: cluster labels as strings or ints

# Subset the data to exclude those clusters
 = adata_2_cleancr[~adata_2_cleancr.obs["donor_id"].isin(clusters_to_remove)].copy()

In [ ]:
adata_2_cleancr

In [ ]:
# Define clusters to remove
clusters_to_remove = ["HPAP109"]  # Example: cluster labels as strings or ints

# Subset the data to exclude those clusters
adata_2_clean = adata_2_clean[~adata_2_clean.obs["donor_id"].isin(clusters_to_remove)].copy()

In [ ]:
import doubletdetection
%matplotlib inline

In [ ]:
clf = doubletdetection.BoostClassifier(
    n_iters=10, 
    clustering_algorithm="leiden", 
    standard_scaling=True,
    pseudocount=0.1,
    n_jobs=-1,
)
doublets = clf.fit(pancdb_adata_both.X).predict(p_thresh=1e-16, voter_thresh=0.5)
doublet_score = clf.doublet_score()

In [ ]:
pancdb_adata_both.obs["doublet"] = doublets
pancdb_adata_both.obs["doublet_score"] = doublet_score

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# DATA
# =========================
scores = pancdb_adata_both.obs["doublet_score"].dropna()

# =========================
# STATS
# =========================
mean_val = scores.mean()
median_val = scores.median()
std_val = scores.std()

print(f"Mean: {mean_val:.3f}")
print(f"Median: {median_val:.3f}")
print(f"Std: {std_val:.3f}")

# =========================
# PLOT
# =========================
plt.figure(figsize=(2, 1.5))

plt.hist(
    scores,
    bins=150,
    range=(0, 10),   # zoom to meaningful range
    edgecolor="black"
)

# -------------------------
# annotate stats
# -------------------------
textstr = (
    f"Mean = {mean_val:.2f}\n"
    f"Median = {median_val:.2f}\n"
    f"SD = {std_val:.2f}"
)

plt.text(
    0.98, 0.95,
    textstr,
    transform=plt.gca().transAxes,
    ha='right',
    va='top'
)

plt.xlabel("Doublet score")
plt.ylabel("Cell count")
plt.title("Doublet score distribution")

plt.tight_layout()

# =========================
# EXPORT
# =========================
out = "/Users/kmeneses/Desktop/Figure4_plots/SI/doublet_score_hist_stats_T2Dpancdb"

plt.savefig(f"{out}.pdf", bbox_inches="tight")
plt.savefig(f"{out}.svg", bbox_inches="tight")

plt.show()

In [ ]:
sc.pl.embedding(pancdb_adata_both, basis='X_umap', ncols=2, cmap='cool', color=["doublet", "doublet_score", "cell_type", "age"])


In [ ]:
pancdb_adata_clean = pancdb_adata_both[pancdb_adata_both.obs["doublet"] != 1].copy()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================
# STYLE (Illustrator-friendly)
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =========================
# COUNT CELLS PER SAMPLE
# =========================
sample_counts = (
    pancdb_adata_clean.obs["donor_id"]
    .value_counts()
    .sort_index()   # keeps donor order clean
)

# Convert to dataframe (optional, useful for export)
df_counts = sample_counts.reset_index()
df_counts.columns = ["donor_id", "n_cells"]

# =========================
# PLOT
# =========================
plt.figure(figsize=(4,2.5))

plt.bar(
    df_counts["donor_id"],
    df_counts["n_cells"]
)

plt.xticks(rotation=90, ha="right")
plt.ylabel("Number of cells")
plt.xlabel("Sample")
plt.title("Cells per sample")

plt.tight_layout()

# =========================
# SAVE
# =========================


plt.show()

In [ ]:
sc.pp.normalize_total(pancdb_adata_clean)
sc.pp.log1p(pancdb_adata_clean)

In [ ]:
# Set device to cpu or to gpu (if your torch has been set up correctly to use GPU), for mac you can use either torch.device('mps') or torch.device('cpu')
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# (Optional) Select top variably expressed/accessible features for analysis (other methods besides seurat_v3 available)
feature_list = ccd.ul.select_features(adata_2_cleancr, n_top_features=2000, flavor='seurat_v3')


# If integrating data across batch, simply add the domain_key argument to indicate the batch key in adata.obs
cur_ccd = ccd.Concord(adata=adata_2_cleancr, input_feature=feature_list, use_faiss=False, domain_key='donor_id', device=device) 

# Encode data, saving the latent embedding in adata.obsm['Concord']
cur_ccd.fit_transform(output_key='Concord')

In [ ]:
ccd.ul.run_umap(adata_2_cleancr, source_key='Concord', result_key='Concord_UMAP', n_components=2, n_neighbors=15, min_dist=0.1, metric='euclidean')

# Plot the UMAP embeddings
color_by = ['donor_id', 'age'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    adata_2_cleancr, basis='Concord_UMAP', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=10, legend_loc='on data')

In [ ]:
# Define clusters to remove
clusters_to_remove = ["unknown"]  # Example: cluster labels as strings or ints

# Subset the data to exclude those clusters
adata_2_cleancr= adata_2_cleancr[~adata_2_cleancr.obs["cell_type"].isin(clusters_to_remove)].copy()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import scanpy as sc

# =========================
# STYLE
# =========================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 7,
    "axes.titlesize": 8,
    "axes.labelsize": 7,
    "font.weight": "normal", 
    "axes.titleweight": "normal",   # 🔥 no bold
    "axes.labelweight": "normal", 
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})
plt.rcParams["figure.figsize"] = (2, 2)
# =========================
# PLOT (let scanpy build grid)
# =========================
fig = sc.pl.embedding(
    adata_2_cleancr,
    basis="Concord_UMAP",
    color=['donor_id', 'cell_type'],
    ncols=1,
    size=1,
    wspace=0.1,                 # 🔥 smaller = lighter file
    frameon=True,
    legend_loc='right',
    show=False,
    return_fig=True           # 🔥 IMPORTANT
)

# =========================
# 🔥 RASTERIZE POINTS (KEY FIX)
# =========================
for ax in fig.axes:
    for coll in ax.collections:
        coll.set_rasterized(True)

# =========================
# CLEAN AXES + BORDER
# =========================
for ax in fig.axes:
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.title.set_fontweight("normal")
    ax.xaxis.label.set_fontweight("normal")
    ax.yaxis.label.set_fontweight("normal")
    
    for spine in ax.spines.values():
        spine.set_linewidth(1)

# =========================
# EXPORT
# =========================
out_base = "/Users/kmeneses/Desktop/Figure4_plots/SI/pandc_ht2dadaultadata_2_cleancr_UMAP_db"

fig.savefig(f"{out_base}.pdf", dpi=600, bbox_inches="tight")  # 🔥 BEST for Illustrator
fig.savefig(f"{out_base}.svg", dpi=600, bbox_inches="tight")

plt.show()

In [ ]:
# Define clusters to remove
clusters_to_remove = ["HPAP109"]  # Example: cluster labels as strings or ints

# Subset the data to exclude those clusters
adata_2_cleancr= adata_2_cleancr[~adata_2_cleancr.obs["donor_id"].isin(clusters_to_remove)].copy()

In [ ]:
ccd.ul.run_umap(adata_2_cleancr, source_key='Concord', result_key='Concord_UMAP', n_components=2, n_neighbors=15, min_dist=0.1, metric='euclidean')

# Plot the UMAP embeddings
color_by = ['donor_id', 'age'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    adata_2_cleancr, basis='Concord_UMAP', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=10, legend_loc='on data')

In [ ]:
color_by = ['donor_id', 'cell_type'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    adata_2_cleancr, basis='Concord_UMAP', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=10, legend_loc='on data')

In [ ]:
adata_2_cleancr

In [ ]:
color_by = ['cell_type', 'doublet_score'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    pancdb_adata_clean, basis='Concord_UMAP', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=10, legend_loc='upper right')

In [ ]:
# Define clusters to remove
clusters_to_remove = ["unknown"]  # Example: cluster labels as strings or ints

# Subset the data to exclude those clusters
pancdb_adata_clean= adata_2[~adata_2.obs["cell_type"].isin(clusters_to_remove)].copy()

In [ ]:
color_by = ['cell_type', 'donor_id'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    pancdb_adata_clean, basis='Concord_UMAP', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=5, legend_loc='on data')

In [ ]:
# Define clusters to remove
clusters_to_remove = ["HPAP067"]  # Example: cluster labels as strings or ints

# Subset the data to exclude those clusters
pancdb_adata_clean = pancdb_adata_clean[~pancdb_adata_clean.obs["donor_id"].isin(clusters_to_remove)].copy()

In [ ]:
color_by = ['cell_type', 'donor_id'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    pancdb_adata_clean093, basis='Concord_UMAP', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=1, point_size=5, legend_loc='upper right')

In [ ]:
# Define clusters to remove
clusters_to_remove = ["HPAP067"]  # Example: cluster labels as strings or ints

# Subset the data to exclude those clusters
pancdb_adata_clean093109 = pancdb_adata_clean093[~pancdb_adata_clean093.obs["donor_id"].isin(clusters_to_remove)].copy()

In [ ]:
# Define clusters to remove
clusters_to_remove = ["HPAP027"]  # Example: cluster labels as strings or ints

# Subset the data to exclude those clusters
pancdb_adata_clean093109 = pancdb_adata_clean093109[~pancdb_adata_clean093109.obs["donor_id"].isin(clusters_to_remove)].copy()

In [ ]:
ccd.ul.run_umap(pancdb_adata_clean093109, source_key='Concord', result_key='Concord_UMAP', n_components=2, n_neighbors=15, min_dist=0.1, metric='euclidean')

# Plot the UMAP embeddings
color_by = ['donor_id', 'cell_type'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    pancdb_adata_clean093109, basis='Concord_UMAP', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=10, legend_loc='on data')

In [ ]:
adata

In [ ]:
color_by = ['donor_id', 'cell_type'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    pancdb_adata_clean093109, basis='Concord_UMAP', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=10, legend_loc='right')

In [ ]:
# Define clusters to remove
clusters_to_remove = ["HPAP109"]  # Example: cluster labels as strings or ints

# Subset the data to exclude those clusters
pancdb_adata_clean093109 = pancdb_adata_clean093109[~pancdb_adata_clean093109.obs["donor_id"].isin(clusters_to_remove)].copy()

In [ ]:
ccd.ul.run_umap(pancdb_adata_clean093109, source_key='Concord', result_key='Concord_UMAP', n_components=2, n_neighbors=15, min_dist=0.1, metric='euclidean')

# Plot the UMAP embeddings
color_by = ['donor_id', 'cell_type'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    pancdb_adata_clean093109, basis='Concord_UMAP', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=10, legend_loc='on data')

In [ ]:
pancdb_adata_clean093109.write('/Volumes/KM_2TB_SSD/Public_human_RNAdasets/pancdb_nd_t2d_cleaned.h5ad')

In [ ]:
pancdb_adata_clean093109 = sc.read_h5ad('/Volumes/KM_2TB_SSD/Public_human_RNAdasets/pancdb_nd_t2d_cleaned.h5ad')

In [ ]:
pancdb_adata_clean093109

In [ ]:
# Column containing clusters
CLUSTER_KEY = "cell_type"    # <-- change if needed

# Clusters you want to merge into a separate AnnData
keep_clusters = ["mesenchymal cell", "endothelial cell"]     # <-- EDIT: choose the two clusters you want

# Subset + merge them into a new AnnData
mesenchymal_endopancdb = adata_2_cleancr[adata_2_cleancr.obs[CLUSTER_KEY].astype(str).isin(keep_clusters)].copy()

print("Cells in merged subset:", mesenchymal_endopancdb.n_obs)

In [ ]:
color_by = ['cell_type', 'disease_state'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    mesenchymal_endopancdb, basis='Concord_UMAP', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=10, legend_loc='on data')

In [ ]:
mesenchymal_endopancdb.write('/Volumes/KM_2TB_SSD/Public_human_RNAdasets/pancdbmesendo_nd_t2d_cleaned.h5ad')

In [ ]:
# Column containing clusters
CLUSTER_KEY = "cell_type"    # <-- change if needed

# Clusters you want to merge into a separate AnnData
keep_clusters = ["mesenchymal cell"]     # <-- EDIT: choose the two clusters you want

# Subset + merge them into a new AnnData
mesenchymal = mesenchymal_endopancdb[mesenchymal_endopancdb.obs[CLUSTER_KEY].astype(str).isin(keep_clusters)].copy()

print("Cells in merged subset:", mesenchymal.n_obs)

In [ ]:
mes.write('/Volumes/KM_2TB_SSD/Public_human_RNAdasets/pancdbmesnotfiltered_nd_t2d_cleaned.h5ad')

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

sc.pl.umap(
    mesenchymal,
    color="disease_state",
    show=True
)



In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np

# =========================================================
# SETTINGS
# =========================================================

CELLTYPE_COL = "cell_type"
CONDITION_COL = "disease_state"   # edit if needed

# define labels exactly as they appear in your data
CONTROL_LABEL = "Control"
T2D_LABEL = "T2D"

# =========================================================
# ENDOTHELIAL CELLS
# =========================================================

endo = mesenchymal_endopancdb[
    (mesenchymal_endopancdb.obs[CELLTYPE_COL] == "endothelial cell") &
    (mesenchymal_endopancdb.obs[CONDITION_COL].isin([CONTROL_LABEL, T2D_LABEL]))
].copy()

# run DGE
sc.tl.rank_genes_groups(
    endo,
    groupby=CONDITION_COL,
    groups=[T2D_LABEL],
    reference=CONTROL_LABEL,
    method="wilcoxon",
    use_raw=False
)

# extract results
endo_deg = sc.get.rank_genes_groups_df(
    endo,
    group=T2D_LABEL
)

# save
endo_deg.to_csv(
    "PancDB_Endothelial_T2D_vs_Control_DEG.csv",
    index=False
)

print(endo_deg.head())


# =========================================================
# MESENCHYMAL CELLS
# =========================================================

mes = mesenchymal_endopancdb[
    (mesenchymal_endopancdb.obs[CELLTYPE_COL] == "mesenchymal cell") &
    (mesenchymal_endopancdb.obs[CONDITION_COL].isin([CONTROL_LABEL, T2D_LABEL]))
].copy()

# run DGE
sc.tl.rank_genes_groups(
    mes,
    groupby=CONDITION_COL,
    groups=[T2D_LABEL],
    reference=CONTROL_LABEL,
    method="wilcoxon",
    use_raw=False
)

# extract results
mes_deg = sc.get.rank_genes_groups_df(
    mes,
    group=T2D_LABEL
)

# save
mes_deg.to_csv(
    "PancDB_Mesenchymal_T2D_vs_Control_DEG.csv",
    index=False
)

print(mes_deg.head())

In [ ]:
print(endo_deg.head(50))


In [ ]:
endo.write('/Volumes/KM_2TB_SSD/Public_human_RNAdasets/pancdECsnotfiltered_nd_t2d_cleaned.h5ad')

In [ ]:
print(mes_deg.head(50))


In [ ]:
mes

In [ ]:
color_by = ['S100A6', 'LUM', 'COL6A3', 'DCN', 'RGS5', 'ADIRF', 'C11orf96', 'ADAMTS4', 'TFPI', 'FABP4', 'COL1A2', 'EDNRA' ] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    mes, basis='Concord_UMAP', color_by=color_by, figsize=(10, 5), dpi=600, ncols=4, font_size=6, point_size=5, legend_loc='upper right', vmax=15)

In [ ]:
sc.pp.neighbors(mesenchymalf)
sc.tl.leiden(mesenchymalf, resolution=0.1, key_added='mesenchymal_leiden')

In [ ]:
color_by = ['mesenchymal_leiden', 'doublet_score'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    mesenchymalf, basis='Concord_UMAP', color_by=color_by, figsize=(10, 5), dpi=600, ncols=2, font_size=6, point_size=10, legend_loc='upper right')

In [ ]:
mesenchymalf.obs['mesenchymal_leiden'].value_counts()

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================
# STYLE
# =========================
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["font.family"] = "Arial"

# =========================
# RUN DEG
# =========================
sc.tl.rank_genes_groups(
    mesenchymalf,
    "mesenchymal_leiden",
    use_raw=False
)

# =========================
# PLOT
# =========================
plt.figure(figsize=(3, 2))   # 🔥 force figure

sc.pl.rank_genes_groups(
    mesenchymalf,
    gene_symbols="features",
    n_genes=20,
    show=False
)

# =========================
# GET FIG + SAVE
# =========================
fig = plt.gcf()   # 🔥 ALWAYS works

out_base = "/Users/kimberlymeneses/Desktop/mesenchymal_markersdr"

fig.savefig(f"{out_base}.svg", dpi=600, bbox_inches="tight")
fig.savefig(f"{out_base}.png", dpi=600, bbox_inches="tight")



In [ ]:
remove_clusters = ["6"]
mesenchymalf = mesenchymalf[~mesenchymalf.obs["mesenchymal_leiden"].isin(remove_clusters)].copy()

In [ ]:
mesenchymalf.obs['mesenchymal_leiden'].value_counts()

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib as mpl

# =========================
# STYLE
# =========================
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["font.family"] = "Arial"

# =========================
# RUN DEG
# =========================
sc.tl.rank_genes_groups(
    mesenchymalf,
    "mesenchymal_leiden",
    use_raw=False
)

# =========================
# PLOT
# =========================
plt.figure(figsize=(3, 2))   # 🔥 force figure

sc.pl.rank_genes_groups(
    mesenchymalf,
    gene_symbols="features",
    n_genes=20,
    show=False
)

# =========================
# GET FIG + SAVE
# =========================
fig = plt.gcf()   # 🔥 ALWAYS works

out_base = "/Users/kimberlymeneses/Desktop/mesenchymal_markersdr_T2d"

fig.savefig(f"{out_base}.svg", dpi=600, bbox_inches="tight")
fig.savefig(f"{out_base}.png", dpi=600, bbox_inches="tight")



In [ ]:
color_by = ['S100A6', 'LUM', 'COL6A3', 'DCN', 'RGS5', 'ADIRF', 'C11orf96', 'ADAMTS4', 'TFPI', 'FABP4', 'COL1A2', 'mesenchymal_leiden' ] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    mesenchymalf, basis='X_umap', color_by=color_by, figsize=(10, 5), dpi=600, ncols=4, font_size=6, point_size=5, legend_loc='upper right', vmax=15)

In [ ]:
# Convert to string if cluster labels are integers
mesenchymalf.obs["mesenchymal_leiden"] = mesenchymalf.obs["mesenchymal_leiden"].astype(str)

# Define the clusters to merge and the new label
to_merge = ["0", "2", "3", "4", "5"]
new_label = "MSL1"

# Apply the merge
mesenchymalf.obs["mesenchymal_leiden"] = mesenchymalf.obs["mesenchymal_leiden"].replace(to_merge, new_label)

In [ ]:
sc.tl.rank_genes_groups(mesenchymalf, "mesenchymal_leiden", use_raw=False)
sc.pl.rank_genes_groups(mesenchymalf, gene_symbols='features')

In [ ]:
# Convert to string if cluster labels are integers
mesenchymalf.obs["mesenchymal_leiden"] = mesenchymalf.obs["mesenchymal_leiden"].astype(str)

# Define the clusters to merge and the new label
to_merge = ["1"]
new_label = "MSL2"

# Apply the merge
mesenchymalf.obs["mesenchymal_leiden"] = mesenchymalf.obs["mesenchymal_leiden"].replace(to_merge, new_label)

In [ ]:
color_by = ['mesenchymal_leiden', 'doublet', 'cell_type', 'total_counts', 'disease_state', 'doublet_score'] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    mesenchymalf, basis='Concord_UMAP', color_by=color_by, figsize=(10, 5), dpi=600, ncols=3, font_size=6, point_size=5, legend_loc='upper right')

In [ ]:
import scanpy as sc
import pandas as pd

# ======================================================
# SETTINGS
# ======================================================

CELLTYPE_COL = "mesenchymal_leiden"   # edit if needed
CONDITION_COL = "disease_state"

CONTROL = "Control"
T2D = "T2D"

clusters = ["MSL1", "MSL2"]

# ======================================================
# LOOP THROUGH CLUSTERS
# ======================================================

for clust in clusters:

    print(f"\n======================")
    print(f"Running DEG for: {clust}")
    print(f"======================")

    ad = mesenchymalf[
        (mesenchymalf.obs[CELLTYPE_COL] == clust) &
        (mesenchymalf.obs[CONDITION_COL].isin([CONTROL, T2D]))
    ].copy()

    print(ad)

    # run DEG
    sc.tl.rank_genes_groups(
        ad,
        groupby=CONDITION_COL,
        groups=[T2D],
        reference=CONTROL,
        method="wilcoxon",
        use_raw=False
    )

    # extract
    deg = sc.get.rank_genes_groups_df(
        ad,
        group=T2D
    )

    # optional contamination removal
    contam = [
        "PRSS1","PRSS2","CPA1","CLPS",
        "CTRB1","CTRB2","CELA3A",
        "REG1A","PLA2G1B"
    ]

    deg = deg[
        ~deg["names"].isin(contam)
    ]

    # save
    outname = f"{clust}_T2D_vs_Control_DEG.csv"

    deg.to_csv(outname, index=False)

    print(deg.head(20))

In [ ]:
mesenchymalf.write('/Volumes/KM_2TB_SSD/Public_human_RNAdasets/pancdbmes_subclusteredfiltered_nd_t2d_cleaned.h5ad')

In [ ]:
mesenchymalf= sc.read_h5ad('/Volumes/KM_2TB_SSD/Public_human_RNAdasets/pancdbmesnotfiltered_nd_t2d_cleaned.h5ad')

In [ ]:
mesenchymalf

In [ ]:
color_by = ['cell_type', 'LUM', 'COL6A3', 'DCN', 'RGS5', 'ADIRF', 'C11orf96', 'ADAMTS4', 'TFPI', 'FABP4', 'COL1A2' ] # Choose which variables you want to visualize
ccd.pl.plot_embedding(
    mesenchymalf, basis='Concord_UMAP', color_by=color_by, figsize=(10, 5), dpi=600, ncols=4, font_size=6, point_size=5, legend_loc='upper right', vmax=15)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

# =====================================================
# STYLE (Illustrator-friendly)
# =====================================================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 6,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =====================================================
# GENES / FEATURES
# =====================================================
color_by = [
    'S100A6',
    'LUM',
    'COL6A3',
    'DCN',
    'RGS5',
    'ADIRF',
    'C11orf96',
    'ADAMTS4',
    'TFPI',
    'FABP4',
    'COL1A2',
    'mesenchymal_leiden'
]

# =====================================================
# PLOT
# =====================================================
ccd.pl.plot_embedding(
    mesenchymalf,
    basis='Concord_UMAP',
    color_by=color_by,
    figsize=(5,2.5),
    dpi=600,
    ncols=4,
    font_size=6,
    point_size=5,
    legend_loc='upper right',
    vmax=15
)

# grab current figure
fig = plt.gcf()

# =====================================================
# SAVE HIGH-RES SVG/PDF
# =====================================================
fig.savefig(
    "/Users/kmeneses/Desktop/mesenchymal_umap_markers.svg",
    format="svg",
    dpi=600,
    bbox_inches="tight"
)

fig.savefig(
    "/Users/kmeneses/Desktop/mesenchymal_umap_markers.pdf",
    format="pdf",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib as mpl

# =====================================================
# STYLE
# =====================================================
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 6,
    "axes.linewidth": 1,
    "pdf.fonttype": 42,
    "svg.fonttype": "none"
})

# =====================================================
# FEATURES
# =====================================================
color_by = [
    'S100A6',
    'LUM',
    'COL6A3',
    'DCN',
    'RGS5',
    'ADIRF',
    'C11orf96',
    'ADAMTS4',
    'TFPI',
    'FABP4',
    'COL1A2',
    'mesenchymal_leiden'
]

# =====================================================
# PLOT + SAVE
# =====================================================
ccd.pl.plot_embedding(
    mesenchymalf,
    basis='Concord_UMAP',
    color_by=color_by,
    figsize=(4,2.5),
    dpi=600,
    ncols=4,
    font_size=6,
    point_size=2,
    legend_loc='upper right',
    vmax=15,
    save_path="/Users/kmeneses/Desktop/mesenchymal_umap_markers.svg"
)